In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/cleaned_data.csv
/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/urban_grid_hour.csv
/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/acn_station_hour.csv


# Base Features

In [2]:
acn_final=pd.read_csv('/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/acn_station_hour.csv',parse_dates=["hour"])
urban_hourly=pd.read_csv('/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/urban_grid_hour.csv',parse_dates=["hour"])
cleaned_data=pd.read_csv('/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/cleaned_data.csv',parse_dates=["hour"])

/tmp/ipykernel_16/1865063460.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned_data=pd.read_csv('/kaggle/input/datasets/sahilhussain2410/ev-vehicles-preprocess-ds/preprocess_ds/cleaned_data.csv',parse_dates=["hour"])


In [3]:
model_df = urban_hourly.copy()

model_df = model_df.sort_values(
    ["entity_id", "hour"]
).reset_index(drop=True)

In [4]:
model_df[
    [
        "hour_of_day",
        "day_of_week",
        "is_weekend",
        "month"
    ]
].head()

,hour_of_day,day_of_week,is_weekend,month
0,0,6,1,6
1,1,6,1,6
2,2,6,1,6
3,3,6,1,6
4,4,6,1,6


In [5]:
peak_hours = [0, 1, 5, 6, 7]
model_df["peak_hour_flag"] = (
    model_df["hour_of_day"]
    .isin(peak_hours)
    .astype(int)
)
model_df["peak_hour_flag"].value_counts()

peak_hour_flag
0    140790
1     37050
Name: count, dtype: int64

In [6]:
off_peak_hours = [
    10,11,12,13,14
]
model_df["off_peak_flag"] = (
    model_df["hour_of_day"]
    .isin(off_peak_hours)
    .astype(int)
)

In [7]:
def utilization_zone(x):

    if x < 0.30:
        return "low"

    elif x < 0.50:
        return "normal"

    elif x < 0.60:
        return "high"

    else:
        return "congested"

model_df["utilization_zone"] = (
    model_df["utilization_rate"]
    .apply(utilization_zone)
)


In [8]:
model_df["utilization_zone"].value_counts()

utilization_zone
low          108747
normal        47739
high          11103
congested     10251
Name: count, dtype: int64

In [9]:
model_df["congestion_flag"] = (
    model_df["utilization_rate"] > 0.60
).astype(int)
model_df["high_utilization_flag"] = (
    model_df["utilization_rate"] > 0.50
).astype(int)

In [10]:
model_df["fast_ratio"] = (
    model_df["fast_count"]
    /
    model_df["charger_count"]
)
model_df["slow_ratio"] = (
    model_df["slow_count"]
    /
    model_df["charger_count"]
)
model_df[
    [
        "fast_ratio",
        "slow_ratio"
    ]
].describe()

,fast_ratio,slow_ratio
count,177840.000000,177840.000000
mean,0.123099,0.876901
std,0.256240,0.256240
min,0.000000,0.000000
25%,0.000000,0.909091
50%,0.000000,1.000000
75%,0.090909,1.000000
max,1.000000,1.000000


In [11]:
model_df["cbd_peak_interaction"] = (
    model_df["CBD"]
    *
    model_df["peak_hour_flag"]
)
model_df["cbd_congestion_interaction"] = (
    model_df["CBD"]
    *
    model_df["congestion_flag"]
)
model_df["revenue_per_charger"] = (
    model_df["revenue"]
    /
    model_df["charger_count"]
)
model_df["demand_per_charger"] = (
    model_df["demand_kwh"]
    /
    model_df["charger_count"]
)

In [12]:
model_df.isna().sum().sort_values(
    ascending=False
).head(10)

entity_id             0
hour                  0
demand_kwh            0
avg_occupancy         0
avg_duration_hours    0
price                 0
charger_count         0
fast_count            0
slow_count            0
area                  0
dtype: int64

In [13]:
model_df.shape

(177840, 34)

In [14]:
model_df.head()

,entity_id,hour,demand_kwh,avg_occupancy,avg_duration_hours,price,charger_count,fast_count,slow_count,area,...,off_peak_flag,utilization_zone,congestion_flag,high_utilization_flag,fast_ratio,slow_ratio,cbd_peak_interaction,cbd_congestion_interaction,revenue_per_charger,demand_per_charger
0,102,2022-06-19 00:00:00,50.983333,12.0,0.728333,0.924,30,3,27,0.71,...,0,normal,0,0,0.1,0.9,0,0,1.570287,1.699444
1,102,2022-06-19 01:00:00,52.500000,12.0,0.750000,0.924,30,3,27,0.71,...,0,normal,0,0,0.1,0.9,0,0,1.617000,1.750000
2,102,2022-06-19 02:00:00,52.500000,12.0,0.750000,0.924,30,3,27,0.71,...,0,normal,0,0,0.1,0.9,0,0,1.617000,1.750000
3,102,2022-06-19 03:00:00,52.500000,12.0,0.750000,0.924,30,3,27,0.71,...,0,normal,0,0,0.1,0.9,0,0,1.617000,1.750000
4,102,2022-06-19 04:00:00,52.500000,12.0,0.750000,0.924,30,3,27,0.71,...,0,normal,0,0,0.1,0.9,0,0,1.617000,1.750000


# lag & rolling features

In [15]:
model_df = model_df.sort_values(
    ["entity_id", "hour"]
)

In [16]:
model_df["demand_lag_1"] = (
    model_df
    .groupby("entity_id")["demand_kwh"]
    .shift(1)
)
model_df["demand_lag_24"] = (
    model_df
    .groupby("entity_id")["demand_kwh"]
    .shift(24)
)
model_df["demand_lag_168"] = (
    model_df
    .groupby("entity_id")["demand_kwh"]
    .shift(168)
)

In [17]:
model_df["rolling_mean_3"] = (
    model_df
    .groupby("entity_id")["demand_kwh"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(3)
         .mean()
    )
)
model_df["rolling_mean_24"] = (
    model_df
    .groupby("entity_id")["demand_kwh"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(24)
         .mean()
    )
)

In [18]:
model_df["rolling_std_24"] = (
    model_df
    .groupby("entity_id")["demand_kwh"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(24)
         .std()
    )
)

In [19]:
model_df["revenue_lag_24"] = (
    model_df
    .groupby("entity_id")["revenue"]
    .shift(24)
)
model_df["revenue_rolling_24"] = (
    model_df
    .groupby("entity_id")["revenue"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(24)
         .mean()
    )
)


In [20]:
model_df["price_lag_1"] = (
    model_df
    .groupby("entity_id")["price"]
    .shift(1)
)
model_df["price_change"] = (
    model_df["price"]
    -
    model_df["price_lag_1"]
)
model_df["price_ratio"] = (
    model_df["price"]
    /
    (
        model_df
        .groupby("entity_id")["price"]
        .transform("mean")
    )
)

In [21]:
model_df["util_lag_24"] = (
    model_df
    .groupby("entity_id")["utilization_rate"]
    .shift(24)
)
model_df["util_rolling_24"] = (
    model_df
    .groupby("entity_id")["utilization_rate"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(24)
         .mean()
    )
)

In [22]:
model_df.isna().sum().sort_values(
    ascending=False
).head(20)

demand_lag_168        41496
rolling_mean_24        5928
demand_lag_24          5928
revenue_lag_24         5928
revenue_rolling_24     5928
rolling_std_24         5928
util_lag_24            5928
util_rolling_24        5928
rolling_mean_3          741
demand_lag_1            247
price_lag_1             247
price_change            247
entity_id                 0
hour                      0
demand_kwh                0
revenue                   0
dynamic_pricing           0
CBD                       0
la                        0
lon                       0
dtype: int64

In [23]:
model_data = (
    model_df
    .dropna(
        subset=[
            "demand_lag_1",
            "demand_lag_24",
            "rolling_mean_24",
            "rolling_std_24"
        ]
    )
).copy()

In [24]:
model_data.columns

Index(['entity_id', 'hour', 'demand_kwh', 'avg_occupancy',
       'avg_duration_hours', 'price', 'charger_count', 'fast_count',
       'slow_count', 'area', 'lon', 'la', 'CBD', 'dynamic_pricing', 'revenue',
       'utilization_rate', 'hour_of_day', 'day_of_week', 'is_weekend', 'month',
       'entity_type', 'region', 'data_source', 'peak_hour_flag',
       'off_peak_flag', 'utilization_zone', 'congestion_flag',
       'high_utilization_flag', 'fast_ratio', 'slow_ratio',
       'cbd_peak_interaction', 'cbd_congestion_interaction',
       'revenue_per_charger', 'demand_per_charger', 'demand_lag_1',
       'demand_lag_24', 'demand_lag_168', 'rolling_mean_3', 'rolling_mean_24',
       'rolling_std_24', 'revenue_lag_24', 'revenue_rolling_24', 'price_lag_1',
       'price_change', 'price_ratio', 'util_lag_24', 'util_rolling_24'],
      dtype='object')

In [25]:
print(model_data.shape)

model_data.head()

(171912, 47)


,entity_id,hour,demand_kwh,avg_occupancy,avg_duration_hours,price,charger_count,fast_count,slow_count,area,...,rolling_mean_3,rolling_mean_24,rolling_std_24,revenue_lag_24,revenue_rolling_24,price_lag_1,price_change,price_ratio,util_lag_24,util_rolling_24
24,102,2022-06-20 00:00:00,49.0,11.0,0.666667,0.924,30,3,27,0.71,...,49.0,54.847674,10.557311,47.1086,56.342741,0.924,0.0,0.902682,0.4,0.408449
25,102,2022-06-20 01:00:00,49.0,11.0,0.666667,0.924,30,3,27,0.71,...,49.0,54.765035,10.596564,48.5100,56.266382,0.924,0.0,0.902682,0.4,0.407060
26,102,2022-06-20 02:00:00,49.0,11.0,0.666667,0.924,30,3,27,0.71,...,49.0,54.619201,10.653025,48.5100,56.131632,0.924,0.0,0.902682,0.4,0.405671
27,102,2022-06-20 03:00:00,49.0,11.0,0.666667,0.924,30,3,27,0.71,...,49.0,54.473368,10.707116,48.5100,55.996882,0.924,0.0,0.902682,0.4,0.404282
28,102,2022-06-20 04:00:00,49.0,11.0,0.666667,0.924,30,3,27,0.71,...,49.0,54.327535,10.758873,48.5100,55.862132,0.924,0.0,0.902682,0.4,0.402894


# part 3

In [26]:
model_data["target_demand"] = model_data["demand_kwh"]


model_data["target_utilization"] = model_data["utilization_rate"]

In [27]:
feature_cols = [

    # temporal
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "month",
    "peak_hour_flag",

    # infrastructure
    "charger_count",
    "fast_count",
    "slow_count",
    "CBD",
    "dynamic_pricing",

    # demand history
    "demand_lag_1",
    "demand_lag_24",
    "rolling_mean_3",
    "rolling_mean_24",
    "rolling_std_24",

    # utilization
    "util_lag_24",
    "util_rolling_24",

    # pricing
    "price",
    "price_lag_1",
    "price_change",
    "price_ratio",

    # revenue
    "revenue_lag_24",
    "revenue_rolling_24",

    # engineered
    "fast_ratio",
    "slow_ratio",
    "demand_per_charger",
    "revenue_per_charger"
]

In [28]:
model_data = model_data[
    feature_cols
    +
    [
        "target_demand",
        "target_utilization",
        "hour",
        "entity_id"
    ]
]

In [29]:
print(model_data.shape)

model_data.head()

(171912, 31)


,hour_of_day,day_of_week,is_weekend,month,peak_hour_flag,charger_count,fast_count,slow_count,CBD,dynamic_pricing,...,revenue_lag_24,revenue_rolling_24,fast_ratio,slow_ratio,demand_per_charger,revenue_per_charger,target_demand,target_utilization,hour,entity_id
24,0,0,0,6,1,30,3,27,0,0,...,47.1086,56.342741,0.1,0.9,1.633333,1.5092,49.0,0.366667,2022-06-20 00:00:00,102
25,1,0,0,6,1,30,3,27,0,0,...,48.5100,56.266382,0.1,0.9,1.633333,1.5092,49.0,0.366667,2022-06-20 01:00:00,102
26,2,0,0,6,0,30,3,27,0,0,...,48.5100,56.131632,0.1,0.9,1.633333,1.5092,49.0,0.366667,2022-06-20 02:00:00,102
27,3,0,0,6,0,30,3,27,0,0,...,48.5100,55.996882,0.1,0.9,1.633333,1.5092,49.0,0.366667,2022-06-20 03:00:00,102
28,4,0,0,6,0,30,3,27,0,0,...,48.5100,55.862132,0.1,0.9,1.633333,1.5092,49.0,0.366667,2022-06-20 04:00:00,102


In [30]:
model_data.to_csv(
    "model_data.csv",
    index=False
)